# World Happiness Index Prediction

## Load Libraries

In [120]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

## Data Extraction

In [121]:
path = "data"
dataset_list = os.listdir(path)
print(dataset_list)

['2015.csv', '2016.csv', '2017.csv', '2018.csv', '2019.csv']


In [122]:
datasets = [pd.read_csv(path + "/" + dataset) for dataset in dataset_list]
df_2015, df_2016, df_2017, df_2018, df_2019 = datasets

##  Initial EDA

In [123]:
df_2015.head()

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176


In [124]:
df_2015.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 158 entries, 0 to 157
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Country                        158 non-null    object 
 1   Region                         158 non-null    object 
 2   Happiness Rank                 158 non-null    int64  
 3   Happiness Score                158 non-null    float64
 4   Standard Error                 158 non-null    float64
 5   Economy (GDP per Capita)       158 non-null    float64
 6   Family                         158 non-null    float64
 7   Health (Life Expectancy)       158 non-null    float64
 8   Freedom                        158 non-null    float64
 9   Trust (Government Corruption)  158 non-null    float64
 10  Generosity                     158 non-null    float64
 11  Dystopia Residual              158 non-null    float64
dtypes: float64(9), int64(1), object(2)
memory usage: 1

In [125]:
for year, data in zip(range(2015,2020), datasets) :
    print(f'{year}     Column: {data.shape[1]}     Row: {data.shape[0]}')

2015     Column: 12     Row: 158
2016     Column: 13     Row: 157
2017     Column: 12     Row: 155
2018     Column: 9     Row: 156
2019     Column: 9     Row: 156


In [126]:
for data in datasets:
    data.columns = data.columns.str.lower()

In [127]:
for year , data in zip(range(2015,2020 ),datasets):
    print(f'{year}\n {data.columns.tolist()}')
    print("--" * 50)

2015
 ['country', 'region', 'happiness rank', 'happiness score', 'standard error', 'economy (gdp per capita)', 'family', 'health (life expectancy)', 'freedom', 'trust (government corruption)', 'generosity', 'dystopia residual']
----------------------------------------------------------------------------------------------------
2016
 ['country', 'region', 'happiness rank', 'happiness score', 'lower confidence interval', 'upper confidence interval', 'economy (gdp per capita)', 'family', 'health (life expectancy)', 'freedom', 'trust (government corruption)', 'generosity', 'dystopia residual']
----------------------------------------------------------------------------------------------------
2017
 ['country', 'happiness.rank', 'happiness.score', 'whisker.high', 'whisker.low', 'economy..gdp.per.capita.', 'family', 'health..life.expectancy.', 'freedom', 'generosity', 'trust..government.corruption.', 'dystopia.residual']
-----------------------------------------------------------------------

In [128]:
column_map = {
    "happiness rank": "happiness_rank",
    "happiness.rank": "happiness_rank",
    "overall rank": "happiness_rank",
    "happiness score": "happiness_score",
    "happiness.score": "happiness_score",
    "score": "happiness_score",
    "economy (gdp per capita)": "gdp_per_capita",
    "economy..gdp.per.capita.": "gdp_per_capita",
    "gdp per capita": "gdp_per_capita",
    "health (life expectancy)": "life_expectancy",
    "health..life.expectancy.": "life_expectancy",
    "healthy life expectancy": "life_expectancy",
    "trust (government corruption)": "government_corruption",
    "trust..government.corruption.": "government_corruption",
    "perceptions of corruption": "government_corruption",
    "family": "family",
    "social support": "family",
    "freedom": "freedom",
    "freedom to make life choices": "freedom",
    "dystopia residual": "dystopia_residual",
    "dystopia.residual": "dystopia_residual",
    "country or region": "country",
    "country": "country",
    "region": "region",
    "generosity": "generosity",
    "standard error": "standard_error",
    "lower confidence interval": "lower_confidence_interval",
    "upper confidence interval": "upper_confidence_interval",
}

for df in datasets:
    df.rename(columns=column_map, inplace=True)

In [129]:
df = pd.concat([df_2015, df_2016, df_2017, df_2018, df_2019])

In [130]:
print(f"columns: {df.shape[1]}\nrows: {df.shape[0]}")

columns: 16
rows: 782


In [131]:
df.head()

,country,region,happiness_rank,happiness_score,standard_error,gdp_per_capita,family,life_expectancy,freedom,government_corruption,generosity,dystopia_residual,lower_confidence_interval,upper_confidence_interval,whisker.high,whisker.low
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738,NaN,NaN,NaN,NaN
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201,NaN,NaN,NaN,NaN
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204,NaN,NaN,NaN,NaN
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531,NaN,NaN,NaN,NaN
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176,NaN,NaN,NaN,NaN


In [132]:
df.columns

Index(['country', 'region', 'happiness_rank', 'happiness_score',
       'standard_error', 'gdp_per_capita', 'family', 'life_expectancy',
       'freedom', 'government_corruption', 'generosity', 'dystopia_residual',
       'lower_confidence_interval', 'upper_confidence_interval',
       'whisker.high', 'whisker.low'],
      dtype='object')

In [133]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 782 entries, 0 to 155
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   country                    782 non-null    object 
 1   region                     315 non-null    object 
 2   happiness_rank             782 non-null    int64  
 3   happiness_score            782 non-null    float64
 4   standard_error             158 non-null    float64
 5   gdp_per_capita             782 non-null    float64
 6   family                     782 non-null    float64
 7   life_expectancy            782 non-null    float64
 8   freedom                    782 non-null    float64
 9   government_corruption      781 non-null    float64
 10  generosity                 782 non-null    float64
 11  dystopia_residual          470 non-null    float64
 12  lower_confidence_interval  157 non-null    float64
 13  upper_confidence_interval  157 non-null    float64
 14 

In [134]:
df.describe().round()

,happiness_rank,happiness_score,standard_error,gdp_per_capita,family,life_expectancy,freedom,government_corruption,generosity,dystopia_residual,lower_confidence_interval,upper_confidence_interval,whisker.high,whisker.low
count,782.0,782.0,158.0,782.0,782.0,782.0,782.0,781.0,782.0,470.0,157.0,157.0,155.0,155.0
mean,79.0,5.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,2.0,5.0,5.0,5.0,5.0
std,45.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
min,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,3.0,3.0,3.0
25%,40.0,5.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,2.0,4.0,4.0,5.0,4.0
50%,79.0,5.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,2.0,5.0,5.0,5.0,5.0
75%,118.0,6.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,2.0,6.0,6.0,6.0,6.0
max,158.0,8.0,0.0,2.0,2.0,1.0,1.0,1.0,1.0,4.0,7.0,8.0,8.0,7.0


## PreProcessing

In [135]:
df.duplicated().sum()

0

In [136]:
for col in df:
    per = df[col].isnull().sum() / df.shape[0] * 100
    print(f'{col}:     {per:.2f}')

country:     0.00
region:     59.72
happiness_rank:     0.00
happiness_score:     0.00
standard_error:     79.80
gdp_per_capita:     0.00
family:     0.00
life_expectancy:     0.00
freedom:     0.00
government_corruption:     0.13
generosity:     0.00
dystopia_residual:     39.90
lower_confidence_interval:     79.92
upper_confidence_interval:     79.92
whisker.high:     80.18
whisker.low:     80.18


In [137]:
df.drop(columns=["region", "standard_error", "dystopia_residual", "lower_confidence_interval", "upper_confidence_interval",
                 "whisker.high", "whisker.low"], inplace=True, errors="ignore")

In [138]:
df.head()

,country,happiness_rank,happiness_score,gdp_per_capita,family,life_expectancy,freedom,government_corruption,generosity
0,Switzerland,1,7.587,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678
1,Iceland,2,7.561,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630
2,Denmark,3,7.527,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139
3,Norway,4,7.522,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699
4,Canada,5,7.427,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811


In [139]:
# lb = LabelEncoder()
# lb.fit_transform(df_2015["Country"])

## Scaling and Normalizing

## Feature Engineering

## Model Selection

## Test & Evaluate